# Test split — the leakage audit and the fourteen rows

Steps 3, 4 and 5 of `docs/preregistration_test.md`: the leakage audit, the twelve local rows,
and the two commercial rows. Step 2, the spend authorization, is a dated line in `docs/budget.md`
and is checked in section 10 before anything is bought.

This is the run that opens the seal. Sections 1-9 spend nothing and run with no API key in the
kernel; section 10 is the only paid part.

---
## 1 — Host and working tree

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

name, memory.total [MiB], memory.used [MiB], driver_version
NVIDIA GeForce RTX 5090, 32607 MiB, 2 MiB, 595.71.05


In [18]:
from pathlib import Path

%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


In [17]:
!git log --oneline -1 -- docs/preregistration_test.md

88de192 docs: state why Sparse-KNN has no rater cells


In [4]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 43.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 70.7 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 74.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 132.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 141.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 184.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 155.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 218.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 209.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621

In [5]:
# torch 2.12 breaks the pinned-torch ABI these three ship against; the pipeline is text-only.
%pip uninstall -q -y torchvision torchaudio torchcodec

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

NVIDIA GeForce RTX 4060 Laptop GPU  sm_89 torch 2.12.0+cu130 / cuda 13.0


---
## 2 — Run parameters

In [19]:
import hashlib
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timedelta, timezone

import yaml

SPLIT = 'test'
EVAL_FILE = Path('data/splits/test.jsonl')

# (condition, committed config). The three RLSF configs carry their own output.name.
ROWS = [
    ('zeroshot',       'configs/base_qwen.yaml'),
    ('random_fewshot', 'configs/base_qwen.yaml'),
    ('knn_fewshot',    'configs/base_qwen.yaml'),
    ('sparse_knn',     'configs/sparse_knn.yaml'),
    ('afsp_margin',    'configs/base_qwen.yaml'),
    ('afsp_full',      'configs/base_qwen.yaml'),
    ('peft',           'configs/peft_qwen.yaml'),
    ('peft_knn',       'configs/peft_afsp.yaml'),
    ('peft_afsp',      'configs/peft_afsp.yaml'),
    ('peft',           'configs/rlsf_eval_w3_0.0.yaml'),
    ('peft',           'configs/rlsf_eval_w3_2.0.yaml'),
    ('peft',           'configs/rlsf_eval_w3_6.0.yaml'),
]
assert len(ROWS) == 12

In [20]:
HASHES = json.loads(Path('data/splits/hashes.json').read_text(encoding='utf-8'))
TEST_SHA = hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest()
assert TEST_SHA == HASHES['hashes']['test.jsonl'], 'test.jsonl is not the committed split'

SEGMENTS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
TEST_SRC = [r['input'] for r in SEGMENTS]
assert len(SEGMENTS) == HASHES['counts']['final']['test'] == 1322, len(SEGMENTS)
print(f'{len(SEGMENTS)} segments  {TEST_SHA[:16]}')

1322 segments  3e24e90f55e5e530


In [25]:
# Every frozen value the pre-registration's settings table names, read back from the configs.
CFGS = {p: yaml.safe_load(Path(p).read_text(encoding='utf-8')) for _, p in ROWS}

for path, cfg in CFGS.items():
    gen = cfg['generator']
    assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', (path, gen['model'])
    assert (gen['temperature'], gen['top_p']) == (0.0, 1.0), f'{path}: not the locked greedy decoding'
    assert (gen['max_tokens'], gen['seed']) == (1024, 42), (path, gen)
    assert gen['dtype'] == 'bfloat16' and gen['load_in_4bit'] is False, f'{path}: base redefined'

BASE, SPARSE = CFGS['configs/base_qwen.yaml'], CFGS['configs/sparse_knn.yaml']
assert BASE['retrieval']['k'] == 8 and BASE['retrieval']['index_dir'] == 'data/knn_index'
assert (BASE['afsp']['beta'], BASE['afsp']['lambda_style']) == (0.3, 0.75), BASE['afsp']
assert BASE['afsp']['style_target_sigma'] == 1.0 and BASE['afsp']['style_objective'] == 'bandpass'
assert (SPARSE['rarity']['min_df'], SPARSE['rarity']['freeze_n']) == (40, 500), SPARSE['rarity']
assert SPARSE['sparse']['m'] == 4, SPARSE['sparse']

ADAPTERS = {p: c['generator']['adapter_path'] for p, c in CFGS.items() if 'adapter_path' in c['generator']}
assert set(ADAPTERS.values()) == {
    'models/peft_lora_r32_lr2e-4/checkpoint-1358',
    'models/rlsf_grpo_w3_0.0/checkpoint-200',
    'models/rlsf_grpo_w3_2.0/checkpoint-200',
    'models/rlsf_grpo_w3_6.0/checkpoint-100',
}, ADAPTERS
print('\n'.join(f'{p:<34} {a}' for p, a in ADAPTERS.items()))

configs/peft_qwen.yaml             models/peft_lora_r32_lr2e-4/checkpoint-1358
configs/peft_afsp.yaml             models/peft_lora_r32_lr2e-4/checkpoint-1358
configs/rlsf_eval_w3_0.0.yaml      models/rlsf_grpo_w3_0.0/checkpoint-200
configs/rlsf_eval_w3_2.0.yaml      models/rlsf_grpo_w3_2.0/checkpoint-200
configs/rlsf_eval_w3_6.0.yaml      models/rlsf_grpo_w3_6.0/checkpoint-100


In [22]:
import getpass
import logging

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
# The local pass runs key-free, so it cannot spend. Section 10 sets the keys; if you have
# already been there, restart the kernel before re-running this cell.

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)

BUDGET_H = 12
DEADLINE = datetime.now(timezone.utc) + timedelta(hours=BUDGET_H)
print(f'HF_TOKEN set, no rater key present. Deadline {DEADLINE:%Y-%m-%d %H:%M}Z')

HF_TOKEN set, no rater key present. Deadline 2026-08-27 06:29Z


---
## 3 — Opening the seal

The committed configs point at `val.jsonl` and stay that way. Each row runs from a derived
config written under `configs/test/`, identical to its source except for `data.eval_file`.
The derived files are recorded in the manifest by digest, so which bytes generated the test
rows is answerable afterwards.

`SEAL_OPEN` gates every cell below that reads `data/splits/test.jsonl`. Set it by hand.

In [23]:
SEAL_OPEN = True

In [24]:
assert SEAL_OPEN, 'set SEAL_OPEN = True to read the sealed split'

TEST_CFG_DIR = Path('configs/test')
TEST_CFG_DIR.mkdir(parents=True, exist_ok=True)
DERIVED = {}

for path in sorted(CFGS):
    cfg = json.loads(json.dumps(CFGS[path]))  # deep copy; the loaded dicts are reused below
    assert cfg['data']['eval_file'] == 'data/splits/val.jsonl', (path, cfg['data'])
    cfg['data']['eval_file'] = str(EVAL_FILE)
    dst = TEST_CFG_DIR / Path(path).name
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    DERIVED[path] = {'path': str(dst), 'sha256': hashlib.sha256(dst.read_bytes()).hexdigest()}
    print(f'{path:<34} -> {dst}  {DERIVED[path]["sha256"][:12]}')

assert not subprocess.run(['git', 'diff', '--quiet', '--', 'configs'], check=False).returncode, \
    'a committed config was modified; the seal must be opened by derivation, not by editing'

configs/base_qwen.yaml             -> configs/test/base_qwen.yaml  5fd13fc4acee
configs/peft_afsp.yaml             -> configs/test/peft_afsp.yaml  d2449f6d7916
configs/peft_qwen.yaml             -> configs/test/peft_qwen.yaml  5ad66601866c
configs/rlsf_eval_w3_0.0.yaml      -> configs/test/rlsf_eval_w3_0.0.yaml  48a6302f30e6
configs/rlsf_eval_w3_2.0.yaml      -> configs/test/rlsf_eval_w3_2.0.yaml  4ee18a7d6fb1
configs/rlsf_eval_w3_6.0.yaml      -> configs/test/rlsf_eval_w3_6.0.yaml  7328a33e97e4
configs/sparse_knn.yaml            -> configs/test/sparse_knn.yaml  3dedce070242


---
## 4 — Weights and the index

In [13]:
from huggingface_hub import snapshot_download

HF_REPO = 'prnamhr/style-aware-mt-models'
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')
ADAPTER_SHA = {}

for adapter in sorted(set(ADAPTERS.values())):
    a = Path(adapter)
    if not all((a / f).exists() for f in ADAPTER_FILES):
        snapshot_download(HF_REPO, local_dir='.', token=os.environ['HF_TOKEN'],
                          allow_patterns=[f'{adapter}/{f}' for f in ADAPTER_FILES])
    conf = json.loads((a / 'adapter_config.json').read_text(encoding='utf-8'))
    ADAPTER_SHA[adapter] = hashlib.sha256((a / 'adapter_model.safetensors').read_bytes()).hexdigest()
    print(f'{adapter:<44} r={conf["r"]} alpha={conf["lora_alpha"]}  {ADAPTER_SHA[adapter][:12]}')

models/peft_lora_r32_lr2e-4/checkpoint-1358  r=32 alpha=64  ad97c46af852
models/rlsf_grpo_w3_0.0/checkpoint-200       r=32 alpha=64  ef232d93a664
models/rlsf_grpo_w3_2.0/checkpoint-200       r=32 alpha=64  c322802ef930
models/rlsf_grpo_w3_6.0/checkpoint-100       r=32 alpha=64  afbd5a74ca41


In [14]:
t0 = time.perf_counter()
snapshot_download('Qwen/Qwen2.5-7B-Instruct', token=os.environ['HF_TOKEN'], max_workers=8,
                  allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'])
print(f'base cached in {(time.perf_counter() - t0) / 60:.1f} min')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

base cached in 1.3 min


In [15]:
INDEX = Path('data/knn_index')
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
INDEX_REBUILT = not all((INDEX / f).exists() for f in INDEX_FILES)

if INDEX_REBUILT:
    !python3 manage.py build_index --config configs/base_qwen.yaml

INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest() for f in INDEX_FILES}
meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == BASE['retrieval']['embed_model'], meta
assert meta['indexed_side'] == 'source' and meta['n_passages'] == 10860, meta

# The unquarantined pool, the one every val row retrieved from. Section 5 audits it, and
# under the declared trigger does not replace it.
print(f'index {"rebuilt" if INDEX_REBUILT else "transferred"}, {meta["n_passages"]} passages')
for f, digest in INDEX_SHA.items():
    print(f'  {f:16s} {digest[:12]}')

index transferred, 10860 passages
  embeddings.npy   9c282c8042ab
  pairs.jsonl      c48f42980943
  meta.json        b028a2a81f9f


---
## 5 — Step 3: the leakage audit

Diagnostic. Without `--write-quarantine`, which would overwrite the val-only 22-row list at
`data/splits/pool_quarantine.json` with one conditioned on the sealed split.

In [19]:
assert SEAL_OPEN
import sys, hashlib, subprocess
from pathlib import Path

QUARANTINE = Path('data/splits/pool_quarantine.json')
QUARANTINE_SHA = hashlib.sha256(QUARANTINE.read_bytes()).hexdigest()

subprocess.run([sys.executable, 'manage.py', 'leakage',
                '--config', 'configs/sparse_retrieval.yaml',
                '--split', 'test', '--unseal-test'], check=True)

assert hashlib.sha256(QUARANTINE.read_bytes()).hexdigest() == QUARANTINE_SHA, \
    'the val-only quarantine was overwritten'

Auditing 1322 test rows against 10860 pool rows ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6701.32it/s]


  21/1322 eval rows flagged, 28 pool rows implicated -> results/leakage_test.json


In [20]:
LEAK = json.loads(Path('results/leakage_test.json').read_text(encoding='utf-8'))
RATE = LEAK['n_eval_rows_flagged'] / LEAK['n_eval_rows']
TRIGGER = 0.0256  # twice the val rate of 17/1323, declared in docs/preregistration_test.md

print(f'{LEAK["n_eval_rows_flagged"]}/{LEAK["n_eval_rows"]} test segments flagged = {RATE:.2%}')
print(f'{LEAK["n_pool_rows_flagged"]} pool rows flagged, {LEAK["n_flags"]} flags')
print(json.dumps(LEAK['max_cos_histogram'], indent=2))
print(f'\ntrigger {TRIGGER:.2%}: ' + ('FIRED — the sensitivity rerun is owed' if RATE > TRIGGER
      else 'not fired — generation proceeds on the unquarantined pool'))

21/1322 test segments flagged = 1.59%
28 pool rows flagged, 28 flags
{
  "0.00-0.50": 0,
  "0.50-0.70": 0,
  "0.70-0.80": 0,
  "0.80-0.85": 1,
  "0.85-0.90": 661,
  "0.90-0.95": 655,
  "0.95-0.97": 5,
  "0.97-0.99": 0,
  "0.99-1.01": 0
}

trigger 2.56%: not fired — generation proceeds on the unquarantined pool


---
## 6 — The gate

In [21]:
from src.infer.run import (_load_configured_glossary, build_fewshot_user, make_client,
                           order_exemplars, resolve_out_name)
from src.retrieval.retrieve import RetrievalIndex

index = RetrievalIndex(BASE['retrieval']['index_dir'], embed_model=BASE['retrieval']['embed_model'])
STYLE = Path(BASE['prompt']['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(BASE)

PROBE_N = 8
probe_src = TEST_SRC[:PROBE_N]
probe = [build_fewshot_user(s, order_exemplars(ex, BASE['prompt']['ordering']), GLOSSARY)
         for s, ex in zip(probe_src, index.retrieve(probe_src, k=BASE['retrieval']['k']))]

t0 = time.perf_counter()
client = make_client(BASE['generator'])
LOAD_S = time.perf_counter() - t0

t0 = time.perf_counter()
for user in probe:
    client.complete(STYLE, user)
SEG_S = (time.perf_counter() - t0) / PROBE_N

print(f'{LOAD_S:.0f}s base load, {SEG_S:.2f}s per segment at k=8')
print(f'{torch.cuda.max_memory_reserved() / 2**30:.1f} GiB reserved')

Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

8s base load, 0.68s per segment at k=8
16.1 GiB reserved


In [22]:
# One process per row, so the base is loaded twelve times.
pass_h = (len(SEGMENTS) * SEG_S + LOAD_S) / 3600
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{pass_h:.2f} h per row, {pass_h * len(ROWS):.1f} h for {len(ROWS)}, {left_h:.1f} h left')

if pass_h * len(ROWS) > 0.9 * left_h:
    print('\nDoes not fit. Generation resumes per row, so a partial session is recoverable.')
else:
    print('\nFits. Section 7 may start.')

0.25 h per row, 3.0 h for 12, 11.9 h left

Fits. Section 7 may start.


In [23]:
del client, index
torch.cuda.empty_cache()

---
## 7 — Step 4: generation

In [ ]:
from src.infer.run import (_load_configured_glossary, build_fewshot_user, make_client,
                           order_exemplars, resolve_out_name)

assert SEAL_OPEN
TIMING = {}

for cond, src_cfg in ROWS:
    cfg_path = DERIVED[src_cfg]['path']
    name = resolve_out_name(cond, CFGS[src_cfg])
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', cfg_path], check=False)
    assert r.returncode == 0, f'{name} exited {r.returncode}'
    TIMING[name] = {'condition': cond, 'config': cfg_path,
                    'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{name}: {TIMING[name]["seconds"] / 60:.1f} min')

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 162.82it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


resuming zeroshot: 1322/1322 already done
Generating 1322 translations with Qwen/Qwen2.5-7B-Instruct (zeroshot) ...
Wrote outputs/zeroshot_test.jsonl
Usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'cost_usd': 0.0}
zeroshot: 0.2 min
Sampling k=8 random exemplars for 1322 sources ...


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 306.10it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


resuming random_fewshot: 1322/1322 already done
Generating 1322 translations with Qwen/Qwen2.5-7B-Instruct (random_fewshot) ...
Wrote outputs/random_fewshot_test.jsonl
Usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'cost_usd': 0.0}
random_fewshot: 0.1 min
Retrieving k=8 exemplars for 1322 sources (most_similar_last) ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 339/339 [00:00<00:00, 634.31it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


resuming knn_fewshot: 1322/1322 already done
Generating 1322 translations with Qwen/Qwen2.5-7B-Instruct (knn_fewshot) ...
Wrote outputs/knn_fewshot_test.jsonl
Usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'cost_usd': 0.0}
knn_fewshot: 0.3 min
sparse_knn: k=8 as up to 4 rarity + cosine for 1322 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1272.26it/s]


In [16]:
NAMES = list(TIMING)
assert len(NAMES) == 12 and len(set(NAMES)) == 12, NAMES

OUTPUT_SHA = {}
for name in NAMES:
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(SEGMENTS), f'{name}: {len(rows)} rows, expected {len(SEGMENTS)}'
    assert [r['input'] for r in rows] == TEST_SRC, f'{name}: source order differs from test.jsonl'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    OUTPUT_SHA[name] = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f'{name:<16} {len(rows)} rows, {len(blank)} blank {blank[:5]}  {OUTPUT_SHA[name][:12]}')

AssertionError: ['zeroshot', 'random_fewshot', 'knn_fewshot']

---
## 8 — Manifest

`output_sha256` is the same digest the raters and COMET will bind their scores to.

In [12]:
import platform

import peft as peft_lib
import transformers

MANIFEST = {
    'split': SPLIT,
    'eval_file': {'path': str(EVAL_FILE), 'sha256': TEST_SHA, 'n': len(SEGMENTS)},
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                             text=True).stdout.strip(),
    'preregistration': 'docs/preregistration_test.md',
    'rows': {name: {**TIMING[name], 'output_sha256': OUTPUT_SHA[name]} for name in NAMES},
    'derived_configs': DERIVED,
    'adapters': ADAPTER_SHA,
    'index': {'dir': str(INDEX), 'rebuilt_here': INDEX_REBUILT, 'sha256': INDEX_SHA, 'meta': meta},
    'quarantine': {'path': str(QUARANTINE), 'sha256': QUARANTINE_SHA, 'applied': False},
    'leakage': {'flagged': LEAK['n_eval_rows_flagged'], 'n': LEAK['n_eval_rows'],
                'rate': round(RATE, 5), 'trigger': TRIGGER, 'fired': bool(RATE > TRIGGER)},
    'versions': {
        'device': torch.cuda.get_device_name(0),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'transformers': transformers.__version__,
        'peft': peft_lib.__version__,
        'python': platform.python_version(),
    },
}
MANIFEST_PATH = Path('outputs/test_manifest.json')
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps(MANIFEST, indent=2))

/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'NAMES' is not defined

---
## 9 — Seal on the local pass

In [38]:
# 1. Nothing was spent.
for name in NAMES:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, (name, usage)
    print(f'{name:<16} {usage["calls"]} calls, ${usage.get("cost_usd", 0.0):.2f}')

# 2. No rater key was ever present.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), var

# 3. The split is what it was, and nothing outside the test artifacts moved.
assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data'],
                       capture_output=True, text=True).stdout.splitlines()
unexpected = [l for l in dirty if 'test' not in l]
assert not unexpected, unexpected

print(f'\nsealed: 0 paid calls, {len(NAMES)} local test rows, no val artifact touched')

zeroshot         1322 calls, $0.00
random_fewshot   1322 calls, $0.00
knn_fewshot      1322 calls, $0.00
sparse_knn       1322 calls, $0.00
afsp_margin      1322 calls, $0.00
afsp_full        1322 calls, $0.00
peft             1322 calls, $0.00
peft_knn         1322 calls, $0.00
peft_afsp        1322 calls, $0.00
rlsf_w3_0.0      1322 calls, $0.00
rlsf_w3_2.0      1322 calls, $0.00
rlsf_w3_6.0      1322 calls, $0.00

sealed: 0 paid calls, 12 local test rows, no val artifact touched


---
## 10 — Step 5: the two commercial rows

The only paid part. `PAID` names them, `SPEND_OK` gates them, and `AUTHORIZED_USD` has to
cover the projection before either runs.

In [34]:
PAID = [
    ('zeroshot',   'configs/commercial_haiku_zeroshot.yaml',      4.774e-4),
    ('sparse_knn', 'configs/commercial_gpt56_sparse_knn.yaml',    7.774e-3),
]
PILOT_N = 20

PAID_CFGS = {p: yaml.safe_load(Path(p).read_text(encoding='utf-8')) for _, p, _ in PAID}
HAIKU = PAID_CFGS['configs/commercial_haiku_zeroshot.yaml']
GPT = PAID_CFGS['configs/commercial_gpt56_sparse_knn.yaml']

assert HAIKU['generator']['model'] == 'claude-haiku-4-5', HAIKU['generator']
assert HAIKU['generator']['thinking'] is False and HAIKU['generator']['temperature'] == 0.0
assert GPT['generator']['model'] == 'gpt-5.6-sol', GPT['generator']
assert GPT['generator']['reasoning_effort'] == 'none', 'thinking would change what the row measures'
assert GPT['retrieval']['k'] == 8 and GPT['retrieval']['index_dir'] == 'data/knn_index'
assert (GPT['rarity']['min_df'], GPT['rarity']['freeze_n']) == (40, 500), GPT['rarity']
assert GPT['sparse']['m'] == 4, GPT['sparse']

PROJECTED = {resolve_out_name(c, PAID_CFGS[p]): rate * len(SEGMENTS) for c, p, rate in PAID}
PROJECTED_TOTAL = sum(PROJECTED.values())
for name, usd in PROJECTED.items():
    print(f'{name:<20} {len(SEGMENTS)} calls  ${usd:.2f}')
print(f'{"total":<20} {2 * len(SEGMENTS)} calls  ${PROJECTED_TOTAL:.2f}')

commercial_haiku     1322 calls  $0.63
gpt56_sparse_knn     1322 calls  $10.28
total                2644 calls  $10.91


In [35]:
SPEND_OK = True
AUTHORIZED_USD = 15

In [36]:
assert SEAL_OPEN and SPEND_OK, 'set SPEND_OK = True to buy the two commercial rows'
assert AUTHORIZED_USD >= PROJECTED_TOTAL, (
    f'${AUTHORIZED_USD:.2f} authorized against a ${PROJECTED_TOTAL:.2f} projection'
)
print(subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short', '--',
                      'docs/budget.md'], capture_output=True, text=True).stdout)
print('the authorization this run spends against must already be a dated line in docs/budget.md')

5f597f8 2026-08-20 docs: clean code

the authorization this run spends against must already be a dated line in docs/budget.md


In [40]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
%pip install -q anthropic==0.109.1 openai==2.41.1


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [50]:
# Two derived configs per row: the pilot caps data.limit, the full pass lifts it.
PAID_DERIVED = {}
for cond, src, _ in PAID:
    for tag, limit in (('pilot', PILOT_N), ('full', None)):
        cfg = json.loads(json.dumps(PAID_CFGS[src]))
        assert cfg['data']['eval_file'] == 'data/splits/val.jsonl', src
        cfg['data']['eval_file'] = str(EVAL_FILE)
        cfg['data']['limit'] = limit
        dst = TEST_CFG_DIR / f'{Path(src).stem}.{tag}.yaml'
        dst.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
        PAID_DERIVED[(src, tag)] = str(dst)
        print(f'{dst}  limit={limit}')

configs/test/commercial_haiku_zeroshot.pilot.yaml  limit=20
configs/test/commercial_haiku_zeroshot.full.yaml  limit=None
configs/test/commercial_gpt56_sparse_knn.pilot.yaml  limit=20
configs/test/commercial_gpt56_sparse_knn.full.yaml  limit=None


In [51]:
import shutil

PILOT = {}
for cond, src, rate in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', PAID_DERIVED[(src, 'pilot')]], check=False)
    assert r.returncode == 0, f'{name} pilot exited {r.returncode}'

    sidecar = Path(f'outputs/{name}_{SPLIT}_usage.json')
    shutil.copy(sidecar, sidecar.with_name(f'{name}_{SPLIT}_pilot_usage.json'))
    u = json.loads(sidecar.read_text(encoding='utf-8'))
    # A zero here is an unpriced model, not a free one, and it would make the guard below inert.
    assert u['cost_usd'] > 0, f'{name}: cost_usd is 0; the model has no pricing entry'
    realized = u['cost_usd'] / u['calls']
    PILOT[name] = {'calls': u['calls'], 'cost_usd': u['cost_usd'], 'rate': realized}
    print(f'{name:<20} {u["calls"]} calls  ${u["cost_usd"]:.4f}  ${realized:.3e}/call  '
          f'projected ${rate:.3e}  x{realized / rate:.2f}')

Output name overridden: condition 'zeroshot' -> outputs/commercial_haiku_test.jsonl


Traceback (most recent call last):
  File "/home/prnamhr/projects/Style-Aware-MT/manage.py", line 80, in <module>
    main()
  File "/home/prnamhr/projects/Style-Aware-MT/manage.py", line 76, in main
    module.main()
  File "/home/prnamhr/projects/Style-Aware-MT/src/infer/run.py", line 437, in main
    run(args.condition, cfg, out_name=args.out_name)
  File "/home/prnamhr/projects/Style-Aware-MT/src/infer/run.py", line 354, in run
    raise ValueError(
ValueError: outputs/commercial_haiku_test.jsonl has 1322 records but only 20 eval rows; delete it to regenerate (did `data.limit` shrink?)


AssertionError: commercial_haiku pilot exited 1

In [ ]:
# A rate this far above the projection means the pilot is not what was priced.
for cond, src, rate in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    assert PILOT[name]['rate'] <= 1.25 * rate, (
        f'{name}: ${PILOT[name]["rate"]:.3e}/call is more than 1.25x the projected ${rate:.3e}; '
        f'the full pass would cost about ${PILOT[name]["rate"] * len(SEGMENTS):.2f}'
    )
revised = sum(PILOT[resolve_out_name(c, PAID_CFGS[p])]['rate'] * len(SEGMENTS) for c, p, _ in PAID)
print(f'at the realized rates the full pass is ${revised:.2f} against ${PROJECTED_TOTAL:.2f} '
      f'projected and ${AUTHORIZED_USD:.2f} authorized')
assert revised <= AUTHORIZED_USD, 'the realized rates exceed the authorization'

In [38]:
for cond, src, _ in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', PAID_DERIVED[(src, 'full')]], check=False)
    assert r.returncode == 0, f'{name} exited {r.returncode}'
    TIMING[name] = {'condition': cond, 'config': PAID_DERIVED[(src, 'full')],
                    'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{name}: {TIMING[name]["seconds"] / 60:.1f} min')

Output name overridden: condition 'zeroshot' -> outputs/commercial_haiku_test.jsonl
resuming commercial_haiku: 1322/1322 already done
Generating 1322 translations with claude-haiku-4-5 (zeroshot) ...
Wrote outputs/commercial_haiku_test.jsonl
Usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'cost_usd': 0.0}
commercial_haiku: 0.1 min
sparse_knn: k=8 as up to 4 rarity + cosine for 1322 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 922.21it/s] 


  routes: {'full': 596, 'partial': 625, 'dense': 101}, mean rare slots filled: 2.77
Output name overridden: condition 'sparse_knn' -> outputs/gpt56_sparse_knn_test.jsonl
resuming gpt56_sparse_knn: 1322/1322 already done
Generating 1322 translations with gpt-5.6-sol (sparse_knn) ...
Wrote outputs/gpt56_sparse_knn_test.jsonl
Usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'cost_usd': 0.0}
gpt56_sparse_knn: 0.3 min


In [44]:
# manage.py is not re-run here: the pilot guard refuses a finished output, and a repeated full
# pass would rewrite the usage sidecars with calls: 0.
assert SEAL_OPEN and SPEND_OK

MANIFEST_PATH = Path('outputs/test_manifest.json')
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert MANIFEST['eval_file']['sha256'] == TEST_SHA, 'the manifest was written against another split'

TIMING = {n: {k: v for k, v in r.items() if k != 'output_sha256'}
          for n, r in MANIFEST['rows'].items()}
OUTPUT_SHA = {n: r['output_sha256'] for n, r in MANIFEST['rows'].items()}
NAMES = list(TIMING)
assert len(NAMES) == 12, NAMES
DERIVED = MANIFEST['derived_configs']
QUARANTINE = Path(MANIFEST['quarantine']['path'])
QUARANTINE_SHA = MANIFEST['quarantine']['sha256']

PILOT = {}
for cond, src, rate in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    u = json.loads(Path(f'outputs/{name}_{SPLIT}_pilot_usage.json').read_text(encoding='utf-8'))
    full = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert u['calls'] == PILOT_N and u['cost_usd'] > 0, (name, u)
    assert u['calls'] + full['calls'] == len(SEGMENTS), (name, u['calls'], full['calls'])
    PILOT[name] = {'calls': u['calls'], 'cost_usd': u['cost_usd'],
                   'rate': u['cost_usd'] / u['calls']}

    # Wall clock was not captured; the mtime of the finished output is what the run left behind.
    mtime = Path(f'outputs/{name}_{SPLIT}.jsonl').stat().st_mtime
    TIMING[name] = {'condition': cond, 'config': PAID_DERIVED[(src, 'full')], 'seconds': None,
                    'finished': datetime.fromtimestamp(mtime, timezone.utc).isoformat(),
                    'timing_source': 'output mtime; the kernel was lost before section 11'}
    print(f'{name:<20} pilot {u["calls"]} calls ${u["cost_usd"]:.4f}  '
          f'full {full["calls"]} calls ${full["cost_usd"]:.4f}  '
          f'finished {TIMING[name]["finished"][:19]}Z')


commercial_haiku     pilot 20 calls $0.0093  full 1302 calls $0.6129  finished 2026-08-26T16:21:29Z
gpt56_sparse_knn     pilot 20 calls $0.1633  full 1302 calls $9.7607  finished 2026-08-26T17:10:26Z


In [45]:
SPEND = {}
for cond, src, _ in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(SEGMENTS), f'{name}: {len(rows)} rows, expected {len(SEGMENTS)}'
    assert [r['input'] for r in rows] == TEST_SRC, f'{name}: source order differs from test.jsonl'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    OUTPUT_SHA[name] = hashlib.sha256(path.read_bytes()).hexdigest()

    full = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    SPEND[name] = {'pilot': PILOT[name], 'full': {k: full[k] for k in ('calls', 'cost_usd')},
                   'total_calls': PILOT[name]['calls'] + full['calls'],
                   'total_usd': round(PILOT[name]['cost_usd'] + full['cost_usd'], 4)}
    print(f'{name:<20} {len(rows)} rows, {len(blank)} blank  '
          f'${SPEND[name]["total_usd"]:.4f} over {SPEND[name]["total_calls"]} calls  '
          f'{OUTPUT_SHA[name][:12]}')

TOTAL_USD = round(sum(v['total_usd'] for v in SPEND.values()), 4)
print(f'\npaid this session: ${TOTAL_USD:.4f} against ${AUTHORIZED_USD:.2f} authorized')

commercial_haiku     1322 rows, 0 blank  $0.6222 over 1322 calls  9758395607ad
gpt56_sparse_knn     1322 rows, 0 blank  $9.9240 over 1322 calls  4ea9f9f51580

paid this session: $10.5462 against $15.00 authorized


---
## 11 — Final seal

In [46]:
ALL = list(TIMING)
assert len(ALL) == 14 and len(set(ALL)) == 14, ALL

for name in ALL:
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    assert path.exists() and OUTPUT_SHA[name] == hashlib.sha256(path.read_bytes()).hexdigest(), name

# The twelve local rows are still free; only the two commercial rows cost anything.
for name in NAMES:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, (name, usage)

assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
assert hashlib.sha256(QUARANTINE.read_bytes()).hexdigest() == QUARANTINE_SHA, 'quarantine changed'
dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data'],
                       capture_output=True, text=True).stdout.splitlines()
unexpected = [l for l in dirty if 'test' not in l]
assert not unexpected, unexpected
print(f'{len(ALL)} rows on test, ${TOTAL_USD:.4f} spent, no val artifact touched')

14 rows on test, $10.5462 spent, no val artifact touched


In [47]:
MANIFEST['rows'] = {name: {**TIMING[name], 'output_sha256': OUTPUT_SHA[name]} for name in ALL}
MANIFEST['derived_configs'] = {**DERIVED,
                               **{f'{k[0]}::{k[1]}': v for k, v in PAID_DERIVED.items()}}
MANIFEST['spend'] = {'authorized_usd': AUTHORIZED_USD, 'projected_usd': round(PROJECTED_TOTAL, 4),
                     'actual_usd': TOTAL_USD, 'per_row': SPEND}
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps(MANIFEST['spend'], indent=2))

{
  "authorized_usd": 15,
  "projected_usd": 10.9084,
  "actual_usd": 10.5462,
  "per_row": {
    "commercial_haiku": {
      "pilot": {
        "calls": 20,
        "cost_usd": 0.0093,
        "rate": 0.00046499999999999997
      },
      "full": {
        "calls": 1302,
        "cost_usd": 0.6129
      },
      "total_calls": 1322,
      "total_usd": 0.6222
    },
    "gpt56_sparse_knn": {
      "pilot": {
        "calls": 20,
        "cost_usd": 0.1633,
        "rate": 0.008165
      },
      "full": {
        "calls": 1302,
        "cost_usd": 9.7607
      },
      "total_calls": 1322,
      "total_usd": 9.924
    }
  }
}


In [48]:
!tar -czf test_generation.tar.gz outputs/*_test.jsonl outputs/*_test_usage.json \
    outputs/*_test_pilot_usage.json outputs/test_manifest.json results/leakage_test.json configs/test
!ls -la test_generation.tar.gz
!git status --short outputs results configs

-rw-r--r-- 1 prnamhr prnamhr 2778725 Aug 26 21:39 test_generation.tar.gz
 M outputs/test_manifest.json
